In [ ]:
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from concurrent.futures import ThreadPoolExecutor


url = "https://play.clickhouse.com/play?user=play"

# Получим имена созданных в 2022 году репозиториев

column_names = [
    'file_time', 'event_type', 'actor_login', 'repo_name', 'created_at', 'updated_at',
    'action', 'comment_id', 'body', 'path', 'position', 'line', 'ref', 'ref_type',
    'creator_user_login', 'number', 'title', 'labels', 'state', 'locked',
    'assignee', 'assignees', 'comments', 'author_association', 'closed_at', 'merged_at',
    'merge_commit_sha', 'requested_reviewers', 'requested_teams', 'head_ref',
    'head_sha', 'base_ref', 'base_sha', 'merged', 'mergeable', 'rebaseable',
    'mergeable_state', 'merged_by', 'review_comments', 'maintainer_can_modify',
    'commits', 'additions', 'deletions', 'changed_files', 'diff_hunk', 'original_position',
    'commit_id', 'original_commit_id', 'push_size', 'push_distinct_size', 'member_login',
    'release_tag_name', 'release_name', 'review_state'
]

def get_created_repos_names(year, count, offset_step=1900, limit=2000):
    repos_name_data = pd.Series(name="repo_name")

    # Создаем новую вкладку для каждого запроса
    driver = webdriver.Chrome()
    offset = 0
    
    for i in range(count):
        driver.get(url)
        query_input = driver.find_element(By.ID, 'query')
        select_query = f"""
            SELECT DISTINCT repo_name 
            FROM github_events
            WHERE event_type = 'CreateEvent' 
                AND toYear(created_at) = '{year}'
                AND ref_type = 'repository'
            LIMIT {limit}
            OFFSET {offset}
        """
        query_input.send_keys(select_query)
        query_input.send_keys(Keys.CONTROL, Keys.RETURN)
        time.sleep(2)
        page_source = driver.page_source
        soup = BeautifulSoup(page_source, 'html.parser')
        table = soup.find('table', {'id': 'data-table'})

        if not table:
            break

        for row in table.find_all('tr')[1:]:
            columns = row.find_all('td')[1:]
            row_data = [col.text.strip() for col in columns]
            repos_name_data = pd.concat([repos_name_data, pd.Series(row_data)])
            repos_name_data.drop_duplicates(inplace=True)
        
        if len(repos_name_data) >= count:
            break

        offset += offset_step

    driver.quit()  # Закрываем вкладки
    return repos_name_data


# Сохраняем DataFrame в CSV-файл
year = 2022
get_created_repos_names(2022, 10000).to_csv(f'repos_name_{year}.csv', index=False)


In [9]:
# Получим квартили 0.25 0.5 0.75 распределения количества нужных событий по репозиториям за 2022 год

import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from concurrent.futures import ThreadPoolExecutor


url = "https://play.clickhouse.com/play?user=play"

def process_array_string(array_string):
    try:
        # Удаляем скобки и разделяем строку по запятым
        elements = array_string.strip("[]").split(",")
        
        # Преобразуем каждый элемент в целое число
        array = [float(element) for element in elements]

        array = list(map(int, array))
        
        return array
    except ValueError as e:
        print(f"Ошибка при обработке строки: {e}")
        return None


def get_quartiles_by_event(year, event):
    row_data = []
    driver = webdriver.Chrome()
    driver.get(url)
    query_input = driver.find_element(By.ID, 'query')
    query = f"""
        SELECT
            quantiles(0.5, 0.8, 0.95, 0.99)(events_count) AS quantiles
        FROM
        (
            SELECT
                repo_name,
                count() AS events_count
            FROM
                github_events
            WHERE
                event_type = '{event}'
                AND toYear(created_at) = {year}
            GROUP BY
                repo_name
        )
        """
    query_input.send_keys(query)
    query_input.send_keys(Keys.CONTROL, Keys.RETURN)
    time.sleep(4)
    page_source = driver.page_source
    soup = BeautifulSoup(page_source, 'html.parser')
    table = soup.find('table', {'id': 'data-table'})

    if not table:
        return

    column = table.find_all('tr')[0].find_all('td')[0]
    row_data = process_array_string(column.text.strip())
    
    driver.quit()  # Закрываем вкладки
    return row_data


events = ['PushEvent', 'ForkEvent', 'PullRequestEvent', 'WatchEvent', 'IssuesEvent']
quartiles_by_event = {}

for event in events:
    quartiles_by_event[event] = get_quartiles_by_event(2022, event)

quartiles_by_event

{'PushEvent': [2, 9, 37, 137],
 'ForkEvent': [1, 3, 10, 41],
 'PullRequestEvent': [4, 12, 35, 166],
 'WatchEvent': [1, 3, 13, 81],
 'IssuesEvent': [2, 8, 36, 158]}

In [ ]:
def pushes_to_deciles(pushes):
    if pushes >= quartiles_by_event[0]:
         